# 00 — Data Preparation (17-class)

Xoá 3 class thiếu data khỏi `data/task-B-group-split`:
- **Calocedrus sp** (1 origin → chỉ có train)
- **Cupressus funebris (Cupressus pendula)** (2 origins → không có val)
- **Pinus sp** (2 origins → không có val)

> ⚠️ Data được fix cứng trong tar.xst trên cloud. Notebook này xoá trực tiếp
> trong thư mục đã extract. Nếu cần khôi phục, extract lại từ tar.xst.

In [1]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = Path.cwd().parent.parent.parent

sys.path.insert(0, str(ROOT))
print(f"ROOT: {ROOT}")

ROOT: /workspace/woodbench


In [ ]:
# Colab-only: install dependencies
if Path('/content').exists():
    get_ipython().system(f'pip install -q -r {ROOT}/requirements.txt')

    # Download dataset from HuggingFace if not exists
    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id="ny0g507h3p/woodbench_A", local_dir=str(DATA_DIR))

In [2]:
import shutil
from pathlib import Path
import sys

DATA_ROOT = Path(r"G:\task-B-group-split")

CLASSES_TO_REMOVE = [
    "Calocedrus sp",
    "Cupressus funebris (Cupressus pendula)",
    "Pinus sp",
]

print(f"Data root: {DATA_ROOT}")
print(f"Exists: {DATA_ROOT.exists()}")

Data root: G:\task-B-group-split
Exists: True


In [24]:
# Kiểm tra trạng thái hiện tại trước khi xoá
print("=" * 60)
print("TRẠNG THÁI HIỆN TẠI")
print("=" * 60)

for split in ["train", "val", "test"]:
    split_dir = DATA_ROOT / split
    if not split_dir.exists():
        print(f"  {split}: NOT FOUND")
        continue
    classes = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    n_images = sum(len(list((split_dir / c).glob("*"))) for c in classes)
    print(f"\n  {split}: {len(classes)} classes, {n_images} images")
    for c in classes:
        n = len(list((split_dir / c).glob("*")))
        marker = " SẼ XOÁ" if c in CLASSES_TO_REMOVE else ""
        print(f"    {c:<50s} {n:>4d}{marker}")

TRẠNG THÁI HIỆN TẠI

  train: 20 classes, 3876 images
    Araucaria araucana                                  209
    Callitris columellaris                              151
    Calocedrus rupestris                                191
    Calocedrus sp                                       238 SẼ XOÁ
    Chamaecyparis obtusa                                220
    Cryptomeria japonica                                217
    Cunninghamia lanceolata                             131
    Cupressus funebris (Cupressus pendula)              163 SẼ XOÁ
    Fokienia hodginsii                                  203
    Picea glauca                                        199
    Pinus cernua                                        199
    Pinus dalatensis                                    234
    Pinus krempfii                                      222
    Pinus radiata                                       231
    Pinus sp                                            110 SẼ XOÁ
    Podocarpus pilgeri   

In [25]:
# XOÁ 3 CLASS
print("=" * 60)
print("XOÁ 3 CLASS THIẾU DATA")
print("=" * 60)

deleted_total = 0
for split in ["train", "val", "test"]:
    split_dir = DATA_ROOT / split
    for cls_name in CLASSES_TO_REMOVE:
        cls_dir = split_dir / cls_name
        if cls_dir.exists():
            n_files = len(list(cls_dir.glob("*")))
            shutil.rmtree(cls_dir)
            deleted_total += n_files
            print(f"  Deleted {split}/{cls_name} ({n_files} files)")
        else:
            print(f"   {split}/{cls_name} — not found (already deleted?)")

print(f"\nTổng: {deleted_total} files đã xoá")

XOÁ 3 CLASS THIẾU DATA
  Deleted train/Calocedrus sp (238 files)
  Deleted train/Cupressus funebris (Cupressus pendula) (163 files)
  Deleted train/Pinus sp (110 files)
   val/Calocedrus sp — not found (already deleted?)
   val/Cupressus funebris (Cupressus pendula) — not found (already deleted?)
   val/Pinus sp — not found (already deleted?)
   test/Calocedrus sp — not found (already deleted?)
  Deleted test/Cupressus funebris (Cupressus pendula) (155 files)
  Deleted test/Pinus sp (106 files)

Tổng: 772 files đã xoá


In [28]:
# VERIFY: đúng 17 class trong mỗi split
print("=" * 60)
print("VERIFICATION")
print("=" * 60)

for split in ["train", "val", "test"]:
    split_dir = DATA_ROOT / split
    classes = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    n_images = sum(len(list((split_dir / c).glob("*"))) for c in classes)

    assert len(classes) == 17, f"{split} has {len(classes)} classes, expected 17!"
    for c in CLASSES_TO_REMOVE:
        assert c not in classes, f"{split} still contains {c}!"

    print(f"  {split}: {len(classes)} classes, {n_images} images ")

print("\nDataset ready: 17 classes in all splits")

VERIFICATION
  train: 17 classes, 3365 images 
  val: 17 classes, 903 images 
  test: 17 classes, 532 images 

Dataset ready: 17 classes in all splits


In [29]:
# Bảng phân bố cuối cùng
print("=" * 60)
print("PHÂN BỐ 17-CLASS DATASET")
print("=" * 60)
print(f"{'Class':<45s} {'Train':>5s} {'Val':>5s} {'Test':>5s} {'Total':>5s}")
print("─" * 65)

totals = {"train": 0, "val": 0, "test": 0}
all_classes = sorted([d.name for d in (DATA_ROOT / "train").iterdir() if d.is_dir()])

for cls in all_classes:
    counts = {}
    for split in ["train", "val", "test"]:
        cls_dir = DATA_ROOT / split / cls
        n = len(list(cls_dir.glob("*"))) if cls_dir.exists() else 0
        counts[split] = n
        totals[split] += n
    total = sum(counts.values())
    print(f"  {cls:<43s} {counts['train']:>5d} {counts['val']:>5d} {counts['test']:>5d} {total:>5d}")

print("─" * 65)
print(f"  {'TOTAL':<43s} {totals['train']:>5d} {totals['val']:>5d} {totals['test']:>5d} {sum(totals.values()):>5d}")

PHÂN BỐ 17-CLASS DATASET
Class                                         Train   Val  Test Total
─────────────────────────────────────────────────────────────────
  Araucaria araucana                            209    54    30   293
  Callitris columellaris                        151    29    34   214
  Calocedrus rupestris                          191    57    24   272
  Chamaecyparis obtusa                          220    64    33   317
  Cryptomeria japonica                          217    70    33   320
  Cunninghamia lanceolata                       131    28    27   186
  Fokienia hodginsii                            203    43    39   285
  Picea glauca                                  199    54    25   278
  Pinus cernua                                  199    57    31   287
  Pinus dalatensis                              234    68    32   334
  Pinus krempfii                                222    46    36   304
  Pinus radiata                                 231    69    31   331